# House Price Prediction and Explanation - Comprehensive Analysis
# Ames, Iowa Housing Dataset

**Author:** Data Science Team  
**Date:** November 2025  
**Objective:** Build interpretable machine learning models to predict house prices and provide actionable insights through comprehensive data analysis, feature engineering, and explainability visualizations.

---

## Table of Contents
18. [Generate Result Files for Submission](#section18)

---

In [5]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
import joblib
import pickle
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print("="*80)
print("LOADING DATA FROM NB1 AND NB2 CACHE")
print("="*80)

# Check cache directory
cache_dir = Path('cache')

if not cache_dir.exists():
    print("\n❌ ERROR: Cache directory not found!")
    print("Please run NB1 and NB2 first to generate the required data.")
    raise FileNotFoundError("Cache directory 'cache/' not found. Run NB1 and NB2 first.")

# Load data from NB1
print("\nLoading NB1 outputs...")
train_engineered = pd.read_pickle(cache_dir / 'train_engineered.pkl')
test_engineered = pd.read_pickle(cache_dir / 'test_engineered.pkl')
train_df = pd.read_pickle(cache_dir / 'train_df.pkl')
test_df = pd.read_pickle(cache_dir / 'test_df.pkl')

with open(cache_dir / 'nb1_metadata.pkl', 'rb') as f:
    nb1_metadata = pickle.load(f)

RANDOM_STATE = nb1_metadata['RANDOM_STATE']
np.random.seed(RANDOM_STATE)

print("✓ Loaded from NB1:")
print(f"  - train_engineered: {train_engineered.shape}")
print(f"  - test_engineered: {test_engineered.shape}")
print(f"  - train_df: {train_df.shape}")
print(f"  - test_df: {test_df.shape}")

# Load data from NB2
print("\nLoading NB2 outputs...")
best_xgb = joblib.load(cache_dir / 'best_xgb.pkl')
best_elastic = joblib.load(cache_dir / 'best_elastic.pkl')
scaler = joblib.load(cache_dir / 'scaler.pkl')

X_train = pd.read_pickle(cache_dir / 'X_train.pkl')
X_val = pd.read_pickle(cache_dir / 'X_val.pkl')
y_train = pd.read_pickle(cache_dir / 'y_train.pkl')
y_val = pd.read_pickle(cache_dir / 'y_val.pkl')
X_full = pd.read_pickle(cache_dir / 'X_full.pkl')
y_log = pd.read_pickle(cache_dir / 'y_log.pkl')

# Load test data for submission
X_test_final = pd.read_pickle(cache_dir / 'X_test_final.pkl')
X_test_scaled = pd.read_pickle(cache_dir / 'X_test_scaled.pkl')

with open(cache_dir / 'nb2_metadata.pkl', 'rb') as f:
    nb2_metadata = pickle.load(f)

common_features = nb2_metadata['common_features']

print("✓ Loaded from NB2:")
print(f"  - best_xgb model")
print(f"  - best_elastic model")
print(f"  - scaler")
print(f"  - X_train: {X_train.shape}")
print(f"  - X_val: {X_val.shape}")
print(f"  - y_train: {y_train.shape}")
print(f"  - y_val: {y_val.shape}")
print(f"  - X_test_final: {X_test_final.shape}")
print(f"  - X_test_scaled: {X_test_scaled.shape}")
print(f"  - common_features: {len(common_features)} features")

print("\n" + "="*80)
print("DATA LOADED SUCCESSFULLY - READY FOR SHAP ANALYSIS")
print("="*80)

LOADING DATA FROM NB1 AND NB2 CACHE

Loading NB1 outputs...
✓ Loaded from NB1:
  - train_engineered: (1460, 179)
  - test_engineered: (1459, 178)
  - train_df: (1460, 81)
  - test_df: (1459, 80)

Loading NB2 outputs...
✓ Loaded from NB2:
  - best_xgb model
  - best_elastic model
  - scaler
  - X_train: (1168, 177)
  - X_val: (292, 177)
  - y_train: (1168,)
  - y_val: (292,)
  - X_test_final: (1459, 177)
  - X_test_scaled: (1459, 177)
  - common_features: 177 features

DATA LOADED SUCCESSFULLY - READY FOR SHAP ANALYSIS


<a id="section18"></a>
## 18. Generate Result Files of Submission

In [6]:
import joblib
from pathlib import Path

print("="*80)
print("LOADING SAVED MODELS FOR INFERENCE")
print("="*80)

# Define models directory
models_dir = Path('models')

# Check if models directory exists
if not models_dir.exists():
    print("\n⚠ Models directory not found. Please run the training and saving cells first.")
else:
    # Load all saved objects
    print("\nLoading models and preprocessing objects...")
    
    # Load XGBoost model
    xgb_loaded = joblib.load(models_dir / 'xgboost_model.pkl')
    print("✓ XGBoost model loaded")
    
    # Load Elastic Net model
    elastic_loaded = joblib.load(models_dir / 'elastic_net_model.pkl')
    print("✓ Elastic Net model loaded")
    
    # Load scaler
    scaler_loaded = joblib.load(models_dir / 'scaler.pkl')
    print("✓ Scaler loaded")
    
    # Load feature information
    feature_info_loaded = joblib.load(models_dir / 'feature_info.pkl')
    print("✓ Feature information loaded")
    
    # Load model parameters
    model_params_loaded = joblib.load(models_dir / 'model_parameters.pkl')
    print("✓ Model parameters loaded")
    
    print("\n" + "="*80)
    print("MAKING PREDICTIONS WITH LOADED MODELS")
    print("="*80)
    
# Generate predictions on test set
predictions_elastic_log = elastic_loaded.predict(X_test_scaled)
predictions_xgb_log = xgb_loaded.predict(X_test_final)

# Convert from log scale back to original scale
predictions_elastic = np.expm1(predictions_elastic_log)
predictions_xgb = np.expm1(predictions_xgb_log)

# Ensemble: Average of both models
predictions_ensemble = (predictions_elastic + predictions_xgb) / 2

print("\n" + "="*80)
print("PREDICTIONS GENERATED")
print("="*80)
print(f"Elastic Net predictions: {len(predictions_elastic)}")
print(f"XGBoost predictions: {len(predictions_xgb)}")
print(f"Ensemble predictions: {len(predictions_ensemble)}")

# Create submission files
submission_elastic = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_elastic
})

submission_xgb = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_xgb
})

submission_ensemble = pd.DataFrame({
    'Id': test_df['Id'],
    'SalePrice': predictions_ensemble
})

# Save submissions
submission_elastic.to_csv('./dataset/submission_elastic_net.csv', index=False)
submission_xgb.to_csv('./dataset/submission_xgboost.csv', index=False)
submission_ensemble.to_csv('./dataset/submission_ensemble.csv', index=False)

print("\n✓ Submission files created:")
print("  - submission_elastic_net.csv")
print("  - submission_xgboost.csv")
print("  - submission_ensemble.csv")

# Display sample predictions
print("\n" + "="*80)
print("SAMPLE PREDICTIONS")
print("="*80)
sample_predictions = pd.DataFrame({
    'Id': test_df['Id'].head(10),
    'Elastic_Net': predictions_elastic[:10],
    'XGBoost': predictions_xgb[:10],
    'Ensemble': predictions_ensemble[:10]
})
print(sample_predictions.to_string(index=False))

# Summary statistics of predictions
print("\n" + "="*80)
print("PREDICTION STATISTICS")
print("="*80)
print(f"\nElastic Net:")
print(f"  Mean: ${predictions_elastic.mean():,.2f}")
print(f"  Median: ${np.median(predictions_elastic):,.2f}")
print(f"  Min: ${predictions_elastic.min():,.2f}")
print(f"  Max: ${predictions_elastic.max():,.2f}")

print(f"\nXGBoost:")
print(f"  Mean: ${predictions_xgb.mean():,.2f}")
print(f"  Median: ${np.median(predictions_xgb):,.2f}")
print(f"  Min: ${predictions_xgb.min():,.2f}")
print(f"  Max: ${predictions_xgb.max():,.2f}")

print(f"\nEnsemble:")
print(f"  Mean: ${predictions_ensemble.mean():,.2f}")
print(f"  Median: ${np.median(predictions_ensemble):,.2f}")
print(f"  Min: ${predictions_ensemble.min():,.2f}")
print(f"  Max: ${predictions_ensemble.max():,.2f}")

print("\n✓ All predictions generated successfully!")

LOADING SAVED MODELS FOR INFERENCE

Loading models and preprocessing objects...
✓ XGBoost model loaded
✓ Elastic Net model loaded
✓ Scaler loaded
✓ Feature information loaded
✓ Model parameters loaded

MAKING PREDICTIONS WITH LOADED MODELS

PREDICTIONS GENERATED
Elastic Net predictions: 1459
XGBoost predictions: 1459
Ensemble predictions: 1459

✓ Submission files created:
  - submission_elastic_net.csv
  - submission_xgboost.csv
  - submission_ensemble.csv

SAMPLE PREDICTIONS
  Id   Elastic_Net       XGBoost      Ensemble
1461 109396.982544 119190.421875 114293.702210
1462 148304.296299 164604.156250 156454.226275
1463 174876.882874 185594.312500 180235.597687
1464 191191.997873 190653.906250 190922.952062
1465 190613.843842 189301.421875 189957.632859
1466 175192.734717 174657.703125 174925.218921
1467 189523.125124 176458.015625 182990.570374
1468 165167.122237 170986.046875 168076.584556
1469 202032.281800 182897.703125 192464.992463
1470 116532.486213 124107.492188 120319.989200

P